# Exercise 2B Two dimensional SH wave source-receiver exercise

## 1. problem setting

This notebook solves a two dimensional homogeneous SH (Shear Horizontal) wave source--receiver problem on a rectangular grid. The receiver trace is compared with a 2D Green-function reference.

**Governing equation**

$$\frac{\partial^2 u}{\partial t^2}=c_s^2\left(\frac{\partial^2u}{\partial x^2}+\frac{\partial^2u}{\partial y^2}\right)+s(x,y,t),\qquad c_s=\sqrt{\frac{\mu}{\rho}}.$$

**Boundary and initial conditions**

$$u=0\ \text{on}\ \partial\Omega,\qquad u(x,y,0)=0,\qquad u_t(x,y,0)=0.$$

**Parameter table**

| Symbol | Meaning | Value |
|---|---:|---:|
| $L_x,L_y$ | domain dimensions | 5000 m, 5000 m |
| $n_x,n_y$ | grid nodes | 500, 500 |
| $\rho$ | density | 2800 kg m$^{-3}$ |
| $\mu$ | shear modulus | 30 GPa |
| CFL | explicit stability factor | 0.45 |
| $f_0$ | source frequency parameter | 2.0 Hz |
| $(x_s,y_s)$ | source location | (2500 m, 2500 m) |
| $(x_r,y_r)$ | receiver location | (3500 m, 3500 m) |
| $t_{end}$ | configured final time | 5 s |


In [ ]:
from pathlib import Path
from dataclasses import dataclass
from time import perf_counter
import base64
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from scipy.sparse import diags, eye, kron, csr_matrix, bmat
from scipy.sparse.linalg import factorized, eigsh, cg
from scipy.linalg import lu_factor, lu_solve
from IPython.display import HTML, display
from scipy.fft import dstn, idstn

Plot_COLORS = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#F0E442", "#000000", "#7F7F7F", "#8B4513"]
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.03,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7,
    "axes.labelsize": 7,
    "axes.titlesize": 7,
    "axes.linewidth": 0.7,
    "axes.prop_cycle": plt.cycler(color=Plot_COLORS),
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "legend.fontsize": 6,
    "legend.frameon": False,
    "lines.linewidth": 1.1,
    "lines.markersize": 3,
    "image.cmap": "viridis",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "animation.embed_limit": 100,
    "animation.embed_limit": 100,
})

CASE_ID = "Exercise2B"
ROOT = Path.cwd()
FIG = ROOT / "figures" / CASE_ID
OUT = ROOT / "outputs" / CASE_ID
FIG.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

@dataclass
class Config:
                                                                                      
    Lx: float = 5000.0
    Ly: float = 5000.0
    nx: int = 500
    ny: int = 500
    rho: float = 2800.0
    mu: float = 30e9
    cfl: float = 0.45
    f0: float = 2.0
    t_end: float = 5.00
    nsave: int = 80
    source_x: float = 2500.0
    source_y: float = 2500.0
    receiver_x: float = 3500.0
    receiver_y: float = 3500.0
    @property
    def cs(self):
        return np.sqrt(self.mu / self.rho)

cfg = Config()
x = np.linspace(0.0, cfg.Lx, cfg.nx)
y = np.linspace(0.0, cfg.Ly, cfg.ny)
dx = x[1] - x[0]
dy = y[1] - y[0]
X, Y = np.meshgrid(x, y, indexing="xy")
cs = cfg.cs
isx = int(np.argmin(np.abs(x - cfg.source_x)))
isy = int(np.argmin(np.abs(y - cfg.source_y)))
irx = int(np.argmin(np.abs(x - cfg.receiver_x)))
iry = int(np.argmin(np.abs(y - cfg.receiver_y)))
t0 = 1.0 / cfg.f0                               
t_end = min(cfg.t_end, min(cfg.Lx, cfg.Ly) / cs / 1.9) + t0                        

U0 = np.zeros((cfg.ny, cfg.nx))
V0 = np.zeros_like(U0)

def source_time(t):
    """Ricker wavelet source time function with dominant frequency cfg.f0."""
    t = np.asarray(t, dtype=float)
    a = (np.pi * cfg.f0 * (t - t0)) ** 2
    return (1.0 - 2.0 * a) * np.exp(-a)

def source(t):
    """Discrete point source s(x,y,t) ≈ src(t) δ(x-xs)δ(y-ys), scaled as in the course code."""
    S = np.zeros((cfg.ny, cfg.nx))
    S[isy, isx] = source_time(t) / (dx * dy)
    return S

def analytical_receiver_2d(times, dt_ref=None):
    """Receiver analytical seismogram from the 2D Green's function convolved with the source."""
    times = np.asarray(times, dtype=float)
    if len(times) == 0:
        return np.array([])
    if dt_ref is None:
        dt_ref = min(cfg.cfl / (cs * np.sqrt(1.0 / dx**2 + 1.0 / dy**2)) / 5.0, 5.0e-4)
    tmax = max(float(times.max()), t_end) + 8.0 * dt_ref
    tref = np.arange(0.0, tmax + dt_ref, dt_ref)
    r = np.sqrt((x[irx] - x[isx]) ** 2 + (y[iry] - y[isy]) ** 2)
    travel_time = r / cs
    G = np.zeros_like(tref)
    mask = tref > travel_time
    G[mask] = 1.0 / (2.0 * np.pi * cs**2 * np.sqrt(tref[mask]**2 - travel_time**2))
    conv = np.convolve(G, source_time(tref) * dt_ref)[:len(tref)]
    return np.interp(times, tref, conv)

def exact(t):
                                                                                  
    return np.zeros((cfg.ny, cfg.nx))

W = np.zeros((cfg.ny, cfg.nx))
r_receiver = np.sqrt((x[irx] - x[isx]) ** 2 + (y[iry] - y[isy]) ** 2)


## 2. shared functions


In [ ]:
def Plot_axes(ax, grid=True):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(direction="out", length=3, width=0.6, pad=2)
    if grid:
        ax.grid(True, color="0.88", linewidth=0.45, alpha=0.8)
    return ax

def linf(u, ref):
    return float(np.max(np.abs(np.asarray(u, dtype=float) - np.asarray(ref, dtype=float))))

def choose_snapshot_steps(nsteps, nsave):
    return set(np.unique(np.round(np.linspace(0, nsteps, min(nsave, nsteps + 1))).astype(int)))

def impose_bc_2d(U):
    U = np.asarray(U, float).copy()
    U[0, :] = 0.0
    U[-1, :] = 0.0
    U[:, 0] = 0.0
    U[:, -1] = 0.0
    return U

def laplacian_fd(U):
    L = np.zeros_like(U)
    L[1:-1, 1:-1] = (
        (U[1:-1, :-2] - 2 * U[1:-1, 1:-1] + U[1:-1, 2:]) / dx**2
        + (U[:-2, 1:-1] - 2 * U[1:-1, 1:-1] + U[2:, 1:-1]) / dy**2
    )
    return L

def plot_field(U, fname, title):
    fig, ax = plt.subplots(figsize=(4.6, 3.8))
    im = ax.imshow(U, origin="lower", extent=[x[0], x[-1], y[0], y[-1]], aspect="equal", cmap="RdBu_r")
    ax.plot(x[isx], y[isy], "r*", ms=8, label="source")
    ax.plot(x[irx], y[iry], "k^", ms=5, label="receiver")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(title)
    ax.legend(loc="upper right")
    Plot_axes(ax, grid=False)
    fig.colorbar(im, ax=ax, shrink=0.85)
    fig.tight_layout()
    fig.savefig(fname)
    plt.close(fig)
    return Path(fname)

def plot_snapshots_field_2d(snapshots, times, fname, title):
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    idx = np.unique(np.round(np.linspace(0, len(times) - 1, min(9, len(times)))).astype(int))
    vmax = max(abs(float(snapshots.min())), abs(float(snapshots.max())), 1e-16)
    fig, axes = plt.subplots(3, 3, figsize=(7.2, 6.5), sharex=True, sharey=True)
    panel_labels = list("abcdefghi")
    im = None
    for k, ax in enumerate(axes.flat):
        if k < len(idx):
            j = idx[k]
            im = ax.imshow(snapshots[j], origin="lower", extent=[x[0], x[-1], y[0], y[-1]], aspect="equal", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
            ax.plot(x[isx], y[isy], "r*", ms=6)
            ax.plot(x[irx], y[iry], "k^", ms=4)
            ax.set_title(f"t={times[j]:.3g} s", pad=2)
            Plot_axes(ax, grid=False)
            ax.text(0.03, 0.94, f"({panel_labels[k]})", transform=ax.transAxes, ha="left", va="top", fontsize=7, fontweight="bold")
        else:
            ax.axis("off")
    for ax in axes[-1, :]: ax.set_xlabel("x [m]")
    for ax in axes[:, 0]: ax.set_ylabel("y [m]")
    if im is not None:
        fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75, pad=0.02)
    fig.suptitle(title, y=1.01, fontsize=8)
    fig.savefig(fname)
    plt.close(fig)
    return Path(fname)

def animate_field_2d(snapshots, times, title, save_path=None):
    snapshots = np.asarray(snapshots)
    times = np.asarray(times)
    vmax = max(abs(float(snapshots.min())), abs(float(snapshots.max())), 1e-16)
    fig, ax = plt.subplots(figsize=(4.0, 3.6))
    im = ax.imshow(snapshots[0], origin="lower", extent=[x[0], x[-1], y[0], y[-1]], aspect="equal", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.plot(x[isx], y[isy], "r*", ms=7, label="source")
    ax.plot(x[irx], y[iry], "k^", ms=5, label="receiver")
    ax.set_xlabel("x [m]")
    ax.set_ylabel("y [m]")
    ax.set_title(title)
    ax.legend(loc="upper right")
    time_text = ax.text(0.03, 0.03, "", transform=ax.transAxes, color="k")
    fig.colorbar(im, ax=ax, shrink=0.85)
    def update(i):
        im.set_data(snapshots[i])
        time_text.set_text(f"t = {times[i]:.3g} s")
        return im, time_text
    anim = FuncAnimation(fig, update, frames=len(times), interval=100, blit=True)
    if save_path is not None:
        try:
            anim.save(save_path, writer=PillowWriter(fps=10))
        except Exception as exc:
            print(f"Animation save skipped for {save_path}: {exc}")
    plt.close(fig)
    try:
        return HTML(anim.to_jshtml())
    except Exception:
        return None

def display_method_result_row(method_label, snapshot_path, anim_obj):
    print(method_label)
    print("snapshot figure:", snapshot_path)
    if anim_obj is not None:
        display(anim_obj)

def wave_error_summary(method, snapshots, times, elapsed, dt, nsteps):
    receiver_num = np.asarray(snapshots)[:, iry, irx]
    receiver_exact = analytical_receiver_2d(times)
    abs_error = np.abs(receiver_num - receiver_exact)
    return {
        "method": method,
        "dt_s": dt,
        "steps": int(nsteps),
        "wall_time_s": elapsed,
        "time_per_step_s": elapsed / max(nsteps, 1),
        "final_receiver_abs_error": float(abs_error[-1]),
        "receiver_Linf_error": linf(receiver_num, receiver_exact),
        "receiver_abs_error_history": abs_error,
        "receiver_numerical": receiver_num,
        "receiver_analytical": receiver_exact,
    }

def save_wave_method(method_key, method_label, snapshots, times, elapsed, dt, nsteps, store):
    store[method_key] = wave_error_summary(method_label, snapshots, times, elapsed, dt, nsteps)
    store[method_key]["snapshots"] = snapshots
    store[method_key]["times"] = times
    snapshot_path = plot_snapshots_field_2d(snapshots, times, FIG / f"{method_key}_snapshots.png", f"{method_label}: snapshots")
    anim = animate_field_2d(snapshots, times, f"{method_label}: animation", save_path=FIG / f"{method_key}_animation.gif")
    display_method_result_row(method_label, snapshot_path, anim)

plot_field(source(t0), FIG / "source_location_and_scaling.png", "Discrete point source at source time peak")
results = {}

def sponge_sigma_2d(width_fraction=0.05, strength_factor=3.0):
    width = width_fraction * min(cfg.Lx, cfg.Ly)
    sigma_max = strength_factor * cs / max(width, 1e-12)
    dist_x = np.minimum(X - x[0], x[-1] - X)
    dist_y = np.minimum(Y - y[0], y[-1] - Y)
    dist_to_boundary = np.minimum(dist_x, dist_y)
    r = np.clip((width - dist_to_boundary) / width, 0.0, 1.0)
    return sigma_max * r**2

sigma_sponge_2d = sponge_sigma_2d()

dt_explicit_fdm_eval = cfg.cfl / (cs * np.sqrt(1.0 / dx**2 + 1.0 / dy**2))
nsteps_explicit_fdm_eval = int(np.ceil(t_end / dt_explicit_fdm_eval))
dt_explicit_fdm_eval = t_end / nsteps_explicit_fdm_eval
                                                                                                                    
def fem_mk(n, h):
    # Assemble linear-element mass and stiffness matrices.
    # Keep boundary coupling terms for the interior solve.
    M = np.zeros((n, n)); K = np.zeros((n, n))
    Me = h / 6.0 * np.array([[2, 1], [1, 2]], float); Ke = 1.0 / h * np.array([[1, -1], [-1, 1]], float)
    for e in range(n - 1):
        sl = slice(e, e + 2); M[sl, sl] += Me; K[sl, sl] += Ke
    return csr_matrix(M), csr_matrix(K)

def fem_wave_stable_dt(Mii, Kii, safety=0.45):
    # Estimate the explicit wave time step from the maximum FEM eigenfrequency.
    """
    Stable explicit FEM wave timestep from the generalized eigenproblem
        K phi = lambda M phi.
    Sparse version avoids forming inv(M)K as a dense matrix.
    """
    lambda_max = eigsh(
        Kii,
        k=1,
        M=Mii,
        which="LM",
        return_eigenvectors=False,
    )[0]
    return min(safety * 2.0 / (cs * np.sqrt(lambda_max)), t_end)

t_start = perf_counter()

Mx_fem_base, Kx_fem_base = fem_mk(cfg.nx, dx)
My_fem_base, Ky_fem_base = fem_mk(cfg.ny, dy)
M_fem_base = kron(My_fem_base, Mx_fem_base, format="csr")
K_fem_base = kron(My_fem_base, Kx_fem_base, format="csr") + kron(Ky_fem_base, Mx_fem_base, format="csr")

interior_mask = np.ones((cfg.ny, cfg.nx), dtype=bool)
interior_mask[0, :] = interior_mask[-1, :] = False
interior_mask[:, 0] = interior_mask[:, -1] = False
interior = np.flatnonzero(interior_mask.ravel())

Kii_fem = K_fem_base[interior[:, None], interior].tocsr()
Kbc_fem = np.zeros(len(interior))                                                          

sigma_int = sigma_sponge_2d.ravel()[interior]
Sigma_fem_2d = diags(sigma_int, 0, format="csr")

Mii_fem_consistent = M_fem_base[interior[:, None], interior].tocsr()

M_fem_lumped_diag = np.asarray(M_fem_base.sum(axis=1)).ravel()
Mii_fem_lumped_diag = M_fem_lumped_diag[interior]
Mii_fem_lumped = diags(Mii_fem_lumped_diag, 0, format="csr")

dt_explicit_fem_consistent_eval = fem_wave_stable_dt(Mii_fem_consistent, Kii_fem)

dt_explicit_fem_lumped_eval = fem_wave_stable_dt(Mii_fem_lumped, Kii_fem)
                                                                                                                                             
dt_common = min(dt_explicit_fdm_eval, dt_explicit_fem_consistent_eval, dt_explicit_fem_lumped_eval)
nsteps_common = int(np.ceil(t_end / dt_common))
dt_common = t_end / nsteps_common
dt_explicit_fdm = dt_common
nsteps_explicit_fdm = nsteps_common
dt_explicit_fem_consistent = dt_common                                                                                    
nsteps_explicit_fem_consistent = nsteps_common                                                                 
dt_explicit_fem_lumped = dt_common                                                                                    
nsteps_explicit_fem_lumped = nsteps_common                                                                 
print(f"Common minimum dt = {dt_common:.6e} s, steps = {nsteps_common}")
print(f"FEM explicit dt eval, consistent mass = {dt_explicit_fem_consistent_eval:.6e} s; using common dt = {dt_explicit_fem_consistent:.6e} s, steps = {nsteps_explicit_fem_consistent}")                                                      
print(f"FEM explicit dt eval, lumped mass     = {dt_explicit_fem_lumped_eval:.6e} s; using common dt = {dt_explicit_fem_lumped:.6e} s, steps = {nsteps_explicit_fem_lumped}")

t_end_timer = perf_counter()
print(f"Total elapsed time for FEM shared setup: {t_end_timer - t_start:.3f} s")

## 3. FDM explicit


In [ ]:
def fdm_explicit():
    # Build the finite-difference Laplacian on interior unknowns.
    # Advance with the explicit update using the CFL-limited time step.
    dt_fdm = dt_explicit_fdm
    nsteps_fdm = nsteps_explicit_fdm
    save_steps = choose_snapshot_steps(nsteps_fdm, cfg.nsave)

    nx_i = cfg.nx - 2
    ny_i = cfg.ny - 2
    n = nx_i * ny_i
    Dxx = diags([np.ones(nx_i-1), -2*np.ones(nx_i), np.ones(nx_i-1)], [-1, 0, 1], format="csr") / dx**2
    Dyy = diags([np.ones(ny_i-1), -2*np.ones(ny_i), np.ones(ny_i-1)], [-1, 0, 1], format="csr") / dy**2
    D = kron(eye(ny_i, format="csr"), Dxx, format="csr") + kron(Dyy, eye(nx_i, format="csr"), format="csr")
                       
    u_bc = impose_bc_2d(np.zeros_like(W))

    bc_grid = np.zeros((ny_i, nx_i))
    bc_grid[:, 0] += u_bc[1:-1, 0] / dx**2
    bc_grid[:, -1] += u_bc[1:-1, -1] / dx**2
    bc_grid[0, :] += u_bc[0, 1:-1] / dy**2
    bc_grid[-1, :] += u_bc[-1, 1:-1] / dy**2
    bc = bc_grid.ravel()                                                          

    def pack_int(U):
        return U[1:-1, 1:-1].ravel()               

    def unpack_int(uvec):
        U = np.zeros((cfg.ny, cfg.nx))
        U[1:-1, 1:-1] = uvec.reshape(ny_i, nx_i)            
        return impose_bc_2d(U)

    u = pack_int(impose_bc_2d(U0))
    v = pack_int(V0)
    sigma_i = pack_int(sigma_sponge_2d)

                                                  
                                                       
    a0 = cs**2 * (D @ u) + cs**2 * bc + pack_int(source(0.0)) - sigma_i * v
    u_prev = u - dt_fdm * v + 0.5 * dt_fdm**2 * a0

    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps_fdm + 1):
        t = step * dt_fdm
        if step in save_steps: snapshots.append(unpack_int(u)); times.append(t)
        if step == nsteps_fdm: break
        rhs = (2.0 - sigma_i * dt_fdm) * u - (1.0 - sigma_i * dt_fdm) * u_prev + dt_fdm**2 * (cs**2 * (D @ u) + cs**2 * bc + pack_int(source(t)))
        u_prev, u = u, rhs
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_fdm, nsteps_fdm

snap, tt, elapsed, dt, ns = fdm_explicit()
print(f"2D FDM central difference elapsed time: {elapsed:.6f} s")
save_wave_method("fdm_2d_central_difference", "2D FDM central difference", snap, tt, elapsed, dt, ns, results)


## 4. FDM implicit


In [ ]:
def fdm_implicit(dt_factor=10):
    # Assemble the backward-Euler system matrix for one implicit time step.
    # Solve the linear system at each time level before applying boundary values.
    dt = dt_common                                                                      
    nsteps = nsteps_common
    save_steps = choose_snapshot_steps(nsteps, cfg.nsave)

    nx_i = cfg.nx - 2
    ny_i = cfg.ny - 2
    n = nx_i * ny_i
    Dxx = diags([np.ones(nx_i-1), -2*np.ones(nx_i), np.ones(nx_i-1)], [-1, 0, 1], format="csr") / dx**2
    Dyy = diags([np.ones(ny_i-1), -2*np.ones(ny_i), np.ones(ny_i-1)], [-1, 0, 1], format="csr") / dy**2
    D = kron(eye(ny_i, format="csr"), Dxx, format="csr") + kron(Dyy, eye(nx_i, format="csr"), format="csr")
    bc = np.zeros(n)                                                          

    I = eye(n, format="csr")
    Z = diags([np.zeros(n)], [0], format="csr")
    Sigma = diags([sigma_sponge_2d[1:-1, 1:-1].ravel()], [0], format="csr")
    L = bmat([[Z, I], [cs**2 * D, -Sigma]], format="csr")
    I_state = eye(2 * n, format="csr")
                                                
                                                    

    A = I_state - dt * L
    solve_A = factorized(A.tocsc())

    def pack_int(U):
        return U[1:-1, 1:-1].ravel()

    def unpack_int(uvec):
        U = np.zeros((cfg.ny, cfg.nx))
        U[1:-1, 1:-1] = uvec.reshape(ny_i, nx_i)
        return impose_bc_2d(U)

    y = np.r_[pack_int(impose_bc_2d(U0)), pack_int(V0)]
    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps + 1):
        t = step * dt
        if step in save_steps:
            snapshots.append(unpack_int(y[:n]))
            times.append(t)
        if step == nsteps:
            break

        rhs = y + dt * np.r_[np.zeros(n), cs**2 * bc + pack_int(source(t + dt))]
        y = solve_A(rhs)

    return np.array(snapshots), np.array(times), perf_counter() - tic, dt, nsteps

snap, tt, elapsed, dt, ns = fdm_implicit(dt_factor=10)
print(f"2D FDM backward Euler elapsed time: {elapsed:.6f} s")
save_wave_method("fdm_2d_backward_euler", "2D FDM backward Euler", snap, tt, elapsed, dt, ns, results)


## 5. FDM Crank--Nicolson


In [ ]:
def fdm_crank_nicolson(dt_factor=10):
    # Assemble Crank--Nicolson left and right time-stepping matrices.
    # Use midpoint diffusion/wave weighting for second-order time accuracy.
    dt = dt_common                                                                      
    nsteps = nsteps_common
    save_steps = choose_snapshot_steps(nsteps, cfg.nsave)

    nx_i = cfg.nx - 2
    ny_i = cfg.ny - 2
    n = nx_i * ny_i
    Dxx = diags([np.ones(nx_i-1), -2*np.ones(nx_i), np.ones(nx_i-1)], [-1, 0, 1], format="csr") / dx**2
    Dyy = diags([np.ones(ny_i-1), -2*np.ones(ny_i), np.ones(ny_i-1)], [-1, 0, 1], format="csr") / dy**2
    D = kron(eye(ny_i, format="csr"), Dxx, format="csr") + kron(Dyy, eye(nx_i, format="csr"), format="csr")
    bc = np.zeros(n)                                                          

    I = eye(n, format="csr")
    Z = diags([np.zeros(n)], [0], format="csr")
    Sigma = diags([sigma_sponge_2d[1:-1, 1:-1].ravel()], [0], format="csr")
    L = bmat([[Z, I], [cs**2 * D, -Sigma]], format="csr")
    I_state = eye(2 * n, format="csr")
                                                
                                                    

    A = I_state - 0.5 * dt * L
    B = I_state + 0.5 * dt * L
    solve_A = factorized(A.tocsc())

    def pack_int(U):
        return U[1:-1, 1:-1].ravel()

    def unpack_int(uvec):
        U = np.zeros((cfg.ny, cfg.nx))
        U[1:-1, 1:-1] = uvec.reshape(ny_i, nx_i)
        return impose_bc_2d(U)

    y = np.r_[pack_int(impose_bc_2d(U0)), pack_int(V0)]
    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps + 1):
        t = step * dt
        if step in save_steps:
            snapshots.append(unpack_int(y[:n]))
            times.append(t)
        if step == nsteps:
            break

        rhs = B @ y + 0.5 * dt * (
            np.r_[np.zeros(n), cs**2 * bc + pack_int(source(t))]
            + np.r_[np.zeros(n), cs**2 * bc + pack_int(source(t + dt))]
        )
        y = solve_A(rhs)

    return np.array(snapshots), np.array(times), perf_counter() - tic, dt, nsteps

snap, tt, elapsed, dt, ns = fdm_crank_nicolson(dt_factor=10)
print(f"2D FDM Crank--Nicolson elapsed time: {elapsed:.6f} s")
save_wave_method("fdm_2d_crank_nicolson", "2D FDM Crank--Nicolson", snap, tt, elapsed, dt, ns, results)


## 6. Method 4 — FEM explicit, consistent and lumped mass


In [ ]:
def fem_explicit_consistent_and_lumped_mass(label, lumped=False):
    # Select either the consistent or lumped FEM mass matrix.
    # Use the FEM stability estimate to set the explicit time step.
    if lumped:
        Mii = Mii_fem_lumped
        dt_fem = dt_explicit_fem_lumped
        nsteps_fem = nsteps_explicit_fem_lumped
    else:
        Mii = Mii_fem_consistent
        dt_fem = dt_explicit_fem_consistent
        nsteps_fem = nsteps_explicit_fem_consistent

    def F(t):
        return np.asarray((M_fem_base @ source(t).ravel())[interior]).ravel()
            
    save_steps = choose_snapshot_steps(nsteps_fem, cfg.nsave)
    u = impose_bc_2d(U0).ravel()[interior].copy()
    v = V0.ravel()[interior].copy()

    solve_M = factorized(Mii.tocsc())

    rhs0 = F(0) - cs**2 * (Kii_fem @ u + Kbc_fem) - Mii @ (Sigma_fem_2d @ v)
    u_prev = u - dt_fem * v + 0.5 * dt_fem**2 * solve_M(rhs0)
    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps_fem + 1):
        t = step * dt_fem
        if step in save_steps:
            U = np.zeros_like(W)
            U.ravel()[interior] = u
            snapshots.append(impose_bc_2d(U))
            times.append(t)
        if step == nsteps_fem:
            break
        v_approx = (u - u_prev) / dt_fem
        rhs = F(t) - cs**2 * (Kii_fem @ u + Kbc_fem)  - Mii @ (Sigma_fem_2d @ v_approx)
        un = 2.0 * u - u_prev + dt_fem**2 * solve_M(rhs)
        u_prev, u = u, un

    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_fem, nsteps_fem

for key, label, lumped in [
    ("fem_explicit_consistent", "2D FEM central difference, consistent mass", False),
    ("fem_explicit_lumped", "2D FEM central difference, lumped mass", True),
]:
    snap, tt, elapsed, dt, ns = fem_explicit_consistent_and_lumped_mass(label, lumped=lumped)
    print(f"{label} elapsed time: {elapsed:.6f} s")
    save_wave_method(key, label, snap, tt, elapsed, dt, ns, results)


## 7. Method 5 — FEM backward, consistent and lumped mass


In [ ]:

def fem_backward_consistent_and_lumped_mass(label, lumped=False, dt_factor=10):
    # Select either the consistent or lumped FEM mass matrix.
    # Assemble the backward FEM matrix or block system for implicit stepping.
    if lumped:
        Mii = Mii_fem_lumped
        dt_ref_fem = dt_explicit_fem_lumped
    else:
        Mii = Mii_fem_consistent
        dt_ref_fem = dt_explicit_fem_consistent

    dt_fem = dt_common                                                                                   
    nsteps_fem = nsteps_common
    n = len(interior)
    I = eye(n, format="csr")
                                             
    A = bmat(
        [
            [I, -dt_fem * I],
            [dt_fem * cs**2 * Kii_fem, Mii + dt_fem * (Mii @ Sigma_fem_2d)],
        ],
        format="csc",
    )
    solve_A = factorized(A)

    def F(t):
        return np.asarray((M_fem_base @ source(t).ravel())[interior]).ravel()

    y_u = impose_bc_2d(U0).ravel()[interior].copy()
    y_v = V0.ravel()[interior].copy()
    save_steps = choose_snapshot_steps(nsteps_fem, cfg.nsave)
    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps_fem + 1):
        t = step * dt_fem
        if step in save_steps:
            U = np.zeros_like(U0)
            U.ravel()[interior] = y_u
            U = impose_bc_2d(U)
            snapshots.append(U.copy())
            times.append(t)
        if step == nsteps_fem: break
        rhs_u = y_u
        rhs_v = Mii @ y_v + dt_fem * (F(t + dt_fem) - cs**2 * Kbc_fem)
        
        y = solve_A(np.r_[rhs_u, rhs_v])
        y_u = y[:n]
        y_v = y[n:]
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt_fem, nsteps_fem

for key, label, lumped in [
    ("fem_be_consistent", "2D FEM backward Euler, consistent mass", False),
    ("fem_be_lumped", "2D FEM backward Euler, lumped mass", True),
]:
    snap, tt, elapsed, dt, ns = fem_backward_consistent_and_lumped_mass(label, lumped=lumped)
    print(f"{label} elapsed time: {elapsed:.6f} s")
    save_wave_method(key, label, snap, tt, elapsed, dt, ns, results)


## 8. Method 6 — pseudospectral cosine method for heat or wave equation


In [ ]:
def pseudospectral_cosine_method_for_heat_or_wave_equation():
    # Transform the initial field and source into modal coefficients.
    # Advance spectral modes independently in time before reconstructing the field.
    """2D pseudospectral modal solution using a cosine--cosine basis.

    The cosine basis corresponds to homogeneous Neumann boundary conditions,
    du/dn = 0 on the rectangular boundary. The zero modes are included.
    This version uses the coefficient method, matching the 1A/2A structure.
    """
    mx = np.arange(1, cfg.nx-1)
    my = np.arange(1, cfg.ny-1)
    kx = mx * np.pi / cfg.Lx
    ky = my * np.pi / cfg.Ly
                                              
    omega2 = cs**2 * (ky[:, None]**2 + kx[None, :]**2)

                 
    def coeff_from_field(U):
        A = np.linalg.solve(Phiy, U)
        return np.linalg.solve(Phix, A.T).T

    dt = dt_explicit_fdm
    nsteps = nsteps_explicit_fdm
    save_steps = choose_snapshot_steps(nsteps, cfg.nsave)
    
    def to_modal(U_int):
        return dstn(U_int, type=1, norm="ortho")

    def to_phys(Q):
        return idstn(Q, type=1, norm="ortho")
        
    U = impose_bc_2d(U0)
    V = impose_bc_2d(V0)
    Q = to_modal(U[1:-1, 1:-1])
    Q_t = to_modal(V[1:-1, 1:-1])
    B0 = to_modal(source(0.0)[1:-1, 1:-1])

    Q_dot0 = Q_t
    U_dot0 = to_phys(Q_dot0)
    damping0 = to_modal(sigma_sponge_2d[1:-1, 1:-1] * U_dot0)
    Q_prev = Q - dt * Q_t + 0.5 * dt**2 * (-omega2 * Q - damping0 + B0)

    snapshots = []
    times = []
    tic = perf_counter()

    for step in range(nsteps + 1):
        t = step * dt
        if step in save_steps:
            U_out = np.zeros_like(U0)
            U_out[1:-1, 1:-1] = to_phys(Q)
            U_out = impose_bc_2d(U_out)
            snapshots.append(U_out.copy())
            times.append(t)
        if step == nsteps: break
        B = to_modal(source(t)[1:-1, 1:-1])
        Q_dot = (Q - Q_prev) / dt
        U_dot = to_phys(Q_dot)
        damping_modal = to_modal(sigma_sponge_2d[1:-1, 1:-1] * U_dot)
        Q_new = (2.0 * Q - Q_prev - dt**2 * omega2 * Q - dt**2 * damping_modal + dt**2 * B)
        Q_prev, Q = Q, Q_new
    return np.array(snapshots), np.array(times), perf_counter() - tic, dt, nsteps

snap, tt, elapsed, dt, ns = pseudospectral_cosine_method_for_heat_or_wave_equation()
print(f"2D pseudospectral sine--sine elapsed time: {elapsed:.6f} s")
save_wave_method("pseudospectral_2d_sine_sine", "2D pseudospectral sine--sine modal solution", snap, tt, elapsed, dt, ns, results)

## 10. L2 error


In [ ]:
rows = []                                                                                
for key, v in results.items():
    times = np.asarray(v["times"], dtype=float)
                                                         
    u_num = np.asarray(v["snapshots"])[:, iry, irx]                                                
    u_ref = analytical_receiver_2d(times)
    diff = u_num - u_ref

    l2_t = np.abs(diff)                                                                                                  
    ref_l2_t = np.abs(u_ref)                                                       
    rel_l2_t = l2_t / np.maximum(ref_l2_t, 1e-300)                                             

    receiver_l2_abs = float(np.sqrt(np.trapezoid(diff**2, x=times)))                                             
    receiver_l2_ref = float(np.sqrt(np.trapezoid(u_ref**2, x=times)))                                                      
    receiver_l2_rel = receiver_l2_abs / max(receiver_l2_ref, 1e-300)                                                

    v["green_reference_snapshots"] = u_ref                                          
    v["L2_error_vs_Green"] = l2_t                                          
    v["relative_L2_error_vs_Green"] = rel_l2_t                                          
    v["Linf_error_vs_Green"] = np.abs(diff)                                          

                                                                       
    v["receiver_numerical"] = u_num
    v["receiver_L2_error_vs_green"] = receiver_l2_abs
    v["receiver_relative_L2_error_vs_green"] = receiver_l2_rel
    v["receiver_Linf_error_vs_green"] = float(np.max(np.abs(diff)))
    v["receiver_abs_error_vs_green_at_saved_times"] = l2_t
    v["receiver_pointwise_relative_error_vs_green_at_saved_times"] = rel_l2_t
    v["receiver_green_reference"] = u_ref

    rows.append({
        "method": v["method"],
        "dt": v.get("dt_s", np.nan),                                                    
        "steps": v.get("steps", np.nan),
        "wall time": v.get("wall_time_s", np.nan),
        "final_L2_error_vs_Green": float(l2_t[-1]),
        "relative L2": float(receiver_l2_rel),                                                
        "final relative L2": float(rel_l2_t[-1]),
        "max_L2_error_vs_Green": float(np.max(l2_t)),                                                            
        "max_Linf_error_vs_Green": float(np.max(v["Linf_error_vs_Green"])),
    })

green_l2_summary = pd.DataFrame(rows)[["method", "dt", "steps", "wall time", "relative L2", "max_L2_error_vs_Green", "final relative L2"]].sort_values(
    "relative L2"
)

display(green_l2_summary)

green_l2_summary.to_csv(
    OUT / "exercise2b_receiver_l2_error_vs_green_convolution.csv",
    index=False
)
